# Exploring HydroMT-FIAT Data

This example demonstrates how to explore and visualize the different data types
used by HydroMT-FIAT: **hazard**, **exposure**, and **vulnerability** data.

We will use the built-in test data that ships with HydroMT-FIAT to inspect
the expected formats and contents of each data type.

In [ ]:
# Imports
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
from hydromt import DataCatalog, log

from hydromt_fiat.data import fetch_data

log.initialize_logging()

## Fetch example data

HydroMT-FIAT provides small example datasets for testing and demonstration.
Let's download the global and local test data.

In [ ]:
# Download test datasets
global_dir = fetch_data("global-data")
local_dir = fetch_data("test-build-data")

# Load the data catalogs
dc = DataCatalog(
    data_libs=[
        Path(global_dir, "data_catalog.yml"),
        Path(local_dir, "data_catalog.yml"),
    ]
)
print(f"Available sources: {list(dc.sources.keys())}")

## 1. Hazard Data

Hazard data represents the flood event or risk scenario. It is stored as a
**raster dataset** (typically GeoTIFF or NetCDF) with spatial dimensions (x, y)
and values representing water depth (in meters).

### Expected format
- **Type:** `RasterDataset` (xarray.Dataset)
- **Dimensions:** x, y (spatial grid)
- **Values:** Water depth in meters (or other hazard intensity)
- **CRS:** Any projected or geographic CRS

In [ ]:
# Read hazard data from the catalog
hazard = dc.get_rasterdataset("flood_event")
print("Type:", type(hazard))
print("\nDataset info:")
print(hazard)

In [ ]:
# Visualize the flood map
fig, ax = plt.subplots(figsize=(8, 6))
hazard[list(hazard.data_vars)[0]].plot(ax=ax, cmap="Blues")
ax.set_title("Flood Event - Water Depth (m)")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.tight_layout()
plt.show()

## 2. Exposure Data

Exposure data represents the assets at risk (e.g., buildings, infrastructure).
It is stored as a **vector dataset** (GeoDataFrame) with geometries and attributes.

### Expected format
- **Type:** `GeoDataFrame` (geopandas)
- **Geometry:** Point, Polygon, or MultiPolygon
- **Required columns:**
  - `geometry`: spatial features
  - A column indicating the object/building type (e.g., `building`, `occupancy_type`)
- **CRS:** Any (will be reprojected to match model)

In [ ]:
# Read exposure data from the catalog
exposure = dc.get_geodataframe("osm_buildings")
print("Type:", type(exposure))
print(f"\nNumber of features: {len(exposure)}")
print(f"Geometry type: {exposure.geom_type.unique()}")
print(f"CRS: {exposure.crs}")
print(f"\nColumns: {list(exposure.columns)}")
print("\nHead:")
exposure.head()

In [ ]:
# Visualize exposure data
fig, ax = plt.subplots(figsize=(8, 6))
exposure.plot(ax=ax, column="building", legend=True, markersize=5)
ax.set_title("Exposure - Building Footprints")
plt.tight_layout()
plt.show()

## 3. Vulnerability Data

Vulnerability data defines the relationship between hazard intensity (e.g., water
depth) and the fraction of damage. These are typically **depth-damage curves**.

### Expected format
- **Type:** `DataFrame` (pandas)
- **Structure:** Tabular CSV with:
  - Index or column for depth values (in meters)
  - Columns for each damage function (values: 0.0 to 1.0 damage fraction)
- **Companion:** A linking table that maps object types to curve identifiers

In [ ]:
# Read vulnerability curves from the catalog
vuln_curves = dc.get_dataframe("jrc_curves")
print("Type:", type(vuln_curves))
print(f"\nShape: {vuln_curves.shape}")
print(f"\nColumns (first 10): {list(vuln_curves.columns[:10])}")
print("\nHead:")
vuln_curves.head()

In [ ]:
# Read the linking table
vuln_link = dc.get_dataframe("jrc_curves_link")
print("Linking table:")
vuln_link.head(10)

In [ ]:
# Plot a few vulnerability curves
fig, ax = plt.subplots(figsize=(8, 5))

# Select a few curves to plot
curves_to_plot = vuln_curves.columns[:5]
for col in curves_to_plot:
    ax.plot(vuln_curves.index, vuln_curves[col], label=col)

ax.set_xlabel("Water Depth (m)")
ax.set_ylabel("Damage Fraction")
ax.set_title("Depth-Damage Curves (JRC)")
ax.legend(loc="lower right", fontsize=8)
ax.set_xlim(0, 6)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Region Data

A region defines the spatial extent for model building. It is a simple
polygon geometry (GeoJSON or similar) that clips all data to the area of interest.

In [ ]:
# Read region
region = gpd.read_file(Path(local_dir, "region.geojson"))
print(f"Region CRS: {region.crs}")
print(f"Region bounds: {region.total_bounds}")

# Plot region with exposure overlay
fig, ax = plt.subplots(figsize=(8, 6))
region.boundary.plot(ax=ax, color="red", linewidth=2, label="Region")
exposure.plot(ax=ax, color="blue", alpha=0.5, markersize=2, label="Buildings")
ax.set_title("Model Region with Exposure Data")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

| Data Type | Format | HydroMT Type | Key Properties |
|-----------|--------|--------------|----------------|
| Hazard | GeoTIFF, NetCDF | `RasterDataset` | Spatial grid, water depth values |
| Exposure | GeoPackage, FlatGeobuf, GeoJSON | `GeoDataFrame` | Vector geometries with attributes |
| Vulnerability | CSV | `DataFrame` | Depth-damage curves + linking table |
| Region | GeoJSON, GeoPackage | `GeoDataFrame` | Single polygon boundary |

For more details on each data type, see the
[data section](../docs/user_guide/data/index.rst) of the user guide.